<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Random Club ID

`random.randint(1, club_count)` can return both ends of the range, so every ID from `1` to `5` is possible. To prove it, the runner picks 30 IDs and tallies how often each one appears.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Random Club ID

import random

club_count = 5

# One random club ID, where both 1 and club_count can be picked
chosen_id = random.randint(1, club_count)
print("Club ID picked for review:", chosen_id)

# Pick 30 times and tally every result to show each valid ID can come up
tally = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
for attempt in range(30):
    pick = random.randint(1, club_count)
    tally[pick] += 1

print("Tally after 30 picks:", tally)
print("Any ID outside 1-5?", any(key < 1 or key > club_count for key in tally))

**Explanation:** The picked ID is always 1, 2, 3, 4, or 5, and never 0 or 6. Over 30 picks most IDs show up several times, but random does **not** mean every ID takes a turn before one repeats. Streaks and repeats are normal, and in a short run one ID might not appear at all. With more picks the tally gets closer to even.

## Popcorn Hack 2: Random Club Record

`random.choice` grabs one whole record (a dictionary) from the list, and then each field is read by its key.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Random Club Record

import random

clubs = [
    {
        "club_name": "Robotics Club",
        "category": "STEM",
        "club_code": "C-101"
    },
    {
        "club_name": "Chess Club",
        "category": "Games",
        "club_code": "C-202"
    },
    {
        "club_name": "Jazz Band",
        "category": "Music",
        "club_code": "C-303"
    }
]

chosen_club = random.choice(clubs)

print("Random club picked for review:")
print("  Club name:", chosen_club["club_name"])
print("  Category:", chosen_club["category"])
print("  Club code:", chosen_club["club_code"])

**Explanation:** The output is always one complete record, so the name, category, and code always belong together and never get mixed across clubs. The same club can also be picked twice in a row.

## Popcorn Hack 3: Random Club API Test

Each round picks a random club and a random action, then the action decides which server-style response prints. Five rounds run so different club and action pairs show up.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - Random Club API Test

import random

clubs = [
    {"club_name": "Robotics Club", "club_code": "C-101"},
    {"club_name": "Chess Club", "club_code": "C-202"},
    {"club_name": "Jazz Band", "club_code": "C-303"}
]

actions = ["GET/search", "POST/create", "PUT/update"]

for round_number in range(1, 6):
    club = random.choice(clubs)
    action = random.choice(actions)
    print("Round", round_number, "->", action, "for code", club["club_code"])

    # Send each action to the response that matches it
    if action == "GET/search":
        print("   200 OK: search returned", club["club_name"])
    elif action == "POST/create":
        print("   201 Created: saved", club["club_name"])
    else:
        print("   200 OK: changed details for", club["club_name"])

**Explanation:** The club and the action are chosen independently, so pairs change from run to run (a search on C-101, then an update on C-303, and so on). The `if`/`elif`/`else` chain guarantees each action prints exactly one matching response.

## Popcorn Hack 4 / Homework: Club Registry QA Simulator

This simulator runs `test_count` tests. Each test picks a random club and a random action and then follows backend-style rules, including **duplicate-code rejection** on create. The `registered_codes` list acts as stored data, so creates and removes change what the later tests find. The test logic lives in a `run_test` function that gets called in a loop.

In [ ]:
# CODE_RUNNER: Popcorn Hack 4 / Homework - Club Registry QA Simulator

import random

clubs = [
    {"club_name": "Robotics Club", "category": "STEM", "club_code": "C-101"},
    {"club_name": "Chess Club", "category": "Games", "club_code": "C-202"},
    {"club_name": "Jazz Band", "category": "Music", "club_code": "C-303"}
]

actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]

# This list plays the part of the database, so it changes as tests run
registered_codes = ["C-101", "C-303"]
test_count = 6

def run_test(number):
    club = random.choice(clubs)
    action = random.choice(actions)
    code = club["club_code"]
    name = club["club_name"]
    is_registered = code in registered_codes

    print("Test " + str(number) + ": " + action + " | " + code + " (" + name + ")")

    if action == "GET/search":
        if is_registered:
            print("   RESULT: found, returned the club record")
        else:
            print("   RESULT: not found, no club with that code")
    elif action == "POST/create":
        if is_registered:
            print("   RESULT: REJECTED, duplicate code " + code)
        else:
            registered_codes.append(code)
            print("   RESULT: created, " + code + " is now registered")
    elif action == "PUT/update":
        if is_registered:
            print("   RESULT: updated the registered club")
        else:
            print("   RESULT: cannot update, " + code + " is not registered")
    else:
        if is_registered:
            registered_codes.remove(code)
            print("   RESULT: removed, " + code + " is no longer registered")
        else:
            print("   RESULT: nothing to remove, " + code + " is not registered")

for n in range(1, test_count + 1):
    run_test(n)

print()
print("Codes in storage:", registered_codes)

**What the simulator does:**

- **Random:** every run mixes clubs and actions differently, so the output is never the same twice (there is no `seed` on purpose).
- **Duplicate handling:** a `POST/create` for a code that is already in `registered_codes` gets rejected. A new code is stored, so creating it a second time in the same run counts as a duplicate.
- **Memory between tests:** a remove deletes the code from storage, so a later search or update on it correctly says it is not registered.
- **Both outcomes covered:** every action has a branch for "stored" and for "not stored", so no test prints nothing.

**Sample run** (yours will differ because the choices are random):

```
Test 1: PUT/update | C-101 (Robotics Club)
   RESULT: updated the registered club
Test 2: POST/create | C-202 (Chess Club)
   RESULT: created, C-202 is now registered
Test 3: GET/search | C-101 (Robotics Club)
   RESULT: found, returned the club record
Test 4: POST/create | C-101 (Robotics Club)
   RESULT: REJECTED, duplicate code C-101
Test 5: POST/create | C-101 (Robotics Club)
   RESULT: REJECTED, duplicate code C-101
Test 6: GET/search | C-202 (Chess Club)
   RESULT: found, returned the club record

Codes in storage: ['C-101', 'C-303', 'C-202']
```